# Mini Proyecto Deep Learning - Start-Up basada en IA
#### Máster Universitario en Ciencia de Datos - UV

### Adrián Carrasco Alcalá y Clara Montalvá Barcenilla


## Descripción

El usuario dibuja su edad y un animal, añadiendo opcionalmente una pequeña descripción del mismo y del contexto deseado para la historia. Se crea tanto un dibujo mejorado del animal pensado para colorear como un cuento interactivo que lo tiene como protagonista y que va acompañado de dibujos y de una voz que narra la historia.

#### <u>Inputs</u>
- Dibujo con la edad del niño
- Dibujo de un animal
- Pequeña descripción del animal y planteamiento de la historia hecha por el usuario (opcional).

#### <u>Modelos</u>
- Red Neuronal Convolucional (CNN): Identificar la edad dibujada (entrenada con el conjunto MNIST).
- CLIP: Detectar los animales a partir de los garabatos.
- ControlNet: Mejorar los dibujos de los animales.
- VAE/ControlNet: Crear un dibujo del animal para colorear.
- API Gemini: Generar las partes de la historia.
- Suno: Generar la voz del narrador que lee la historia.
- Stable Difussion: Crear los dibujos a partir de las partes de la historia.
- ControlNet: Combinar los dibujos creados a partir de la historia con los de los animales mejorados.

#### <u>Outputs</u>
- Texto con la historia completa divida en partes, con peticiones de dibujos entre ellas.
- Audio con la voz del narrador contando la historia.
- Imágenes con dibujos creados a partir de la historia y que la complementan.
- Imagen con un dibujo mejorado del animal pintado por el usuario, preparado para colorearlo. 

### Diagrama con modelos de IA

<img src="Diagrama_Proyecto_DL.jpg" width="900" height="500">


## Aplicación de los modelos

### Red Neuronal Convolucional (CNN) para la detección de la edad

Utilizamos una red neuronal entrenada con el conjunto de datos MNIST que sea capaz de identificar el número dibujado por el niño o el padre para clasificarlo en uno de los siguientes grupos de edad:
- 3-5 años
- 6-9 años
- 10-12 años

En primer lugar, abrimos la ventana de dibujo que pregunta por la edad.

In [1]:
import numpy as np
import tensorflow as tf
import cv2
import tkinter as tk
from PIL import Image, ImageDraw
from skimage.transform import resize
from modulos.AplicacionDibujo import AplicacionDibujo

In [2]:
root = tk.Tk()  # Creamos una ventana básica en blanco
app = AplicacionDibujo(root)

# Iniciar el bucle de la ventana (el código se detiene aquí hasta que se cierre la ventana)
root.mainloop()

if app.imagen_resultado is not None:
    print("La imagen ha sido guardada.")
    img = app.imagen_resultado
else:
    print("La ventana se cerró sin guardar la imagen.")

La imagen ha sido guardada.


Cargamos el modelo.

In [3]:
modelo = tf.keras.models.load_model('modelos/CNN_edad.keras')

In [4]:
from modulos.edad import deteccion_edad

edad = deteccion_edad(img, modelo)

print("La edad del niño es:", edad)

La edad del niño es: 7


### Modelo Hugging Face para identificación del dibujo del animal

In [6]:
root = tk.Tk()  # Creamos una ventana básica en blanco
app = AplicacionDibujo(root, modo_edad=False)

# Iniciar el bucle de la ventana (el código se detiene aquí hasta que se cierre la ventana)
root.mainloop()

if app.imagen_resultado is not None:
    print("La imagen ha sido guardada.")
    img_animal = app.imagen_resultado
else:
    print("La ventana se cerró sin guardar la imagen.")

La imagen ha sido guardada.


In [7]:
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor

modelo_hf = "kmewhort/beit-sketch-classifier"
procesador = AutoImageProcessor.from_pretrained(modelo_hf)
modelo = AutoModelForImageClassification.from_pretrained(modelo_hf)

def clasificar_dibujo(matriz_imagen):
    # Transformamos la matriz del dibujo a una imagen PIL
    imagen = Image.fromarray(matriz_imagen.astype('uint8')).convert("RGB")
    
    # Preprocesamos la imagen
    entrada = procesador(images=imagen, return_tensors="pt")

    # Predecimos con el modelo
    with torch.no_grad():
        salida = modelo(**entrada)
        preds = salida.logits

    # Nos quedamos con el índice con la probabilidad más alta
    categoria_predicha = np.argmax(preds).item()
    
    # Dado el identificador de la categoria predicha, buscamos la etiqueta correspondiente
    animal_predicho = modelo.config.id2label[categoria_predicha]
    
    return animal_predicho

animal = clasificar_dibujo(img_animal)
print(f"El animal dibujado es: {animal}")

The image processor of type `BeitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

El animal dibujado es: fish


### API Gemini para generación de la primera parte de la historia

In [8]:
from google import genai

In [9]:
# Leemos la clave API desde un archivo de texto
with open("clave.txt", "r", encoding="utf-8") as archivo:
    GEMINI_API_KEY = archivo.read()

In [ ]:
cliente = genai.Client(api_key=GEMINI_API_KEY)

modelo = "gemini-3-flash-preview"
prompt = f'''Eres un experto escritor de cuentos infantiles. 
Escribe la primera parte de un cuento que tenga a un {animal} como protagonista de la historia. La extensión de esta primera parte debe ser estrictamente de 3 párrafos.
La historia tendrá dos partes, por lo que la trama debe evolucionar rápido. Además, el cuento debe estar escrito específicamente para que lo entienda un niño de {edad} años, utilizando un vocabulario apropiado para esa etapa de desarrollo vital.
Debes tener en cuenta que el texto será leído por un narrador en voz alta, y que se generará una dibujo a partir de él, por lo que debe ser descriptivo y con pausas.
Por último, debe introducirse casi al final de esta primera parte de la historia a un segundo animal desconocido, del cual no se dé una descripción muy detallada para permitir que sea elegido libremente por el niño.
Añade un cuarto párrafo adicional que sea estrictamente una pregunta corta para el niño acerca de cuál es este nuevo animal (la pregunta debe estar en el tiempo verbal de la historia).'''

respuesta = cliente.models.generate_content(
    model = modelo, contents = prompt
)

cuento = respuesta.text

print(cuento)

Había una vez un pequeño pez llamado Burbuja que tenía las escamas de un color naranja tan brillante que parecía un trocito de sol bajo el agua. Burbuja vivía en un arrecife de coral lleno de plantas verdes que bailaban suavemente con la corriente y burbujas que subían a la superficie como globos de cristal. Aunque su hogar era muy bonito, Burbuja siempre miraba hacia el océano azul profundo porque soñaba con vivir una gran aventura fuera de su jardín de rocas.

Una mañana, Burbuja decidió nadar más lejos que nunca y llegó a un lugar donde la arena era blanca como la nieve y el agua brillaba con reflejos plateados. De repente, vio algo que se movía entre las algas gigantes y, con mucha valentía, se acercó para ver qué era. El corazón le latía muy rápido de la emoción mientras esquivaba las corrientes de agua fría, sintiendo que estaba a punto de descubrir el secreto más grande de todo el fondo del mar.

Justo cuando cruzaba un túnel de piedras brillantes, Burbuja se detuvo en seco al v

### Modelo T2S Edge API Microsoft para la generación de la voz del narrador

In [11]:
import asyncio
import os
from modulos.narrador import crear_audiolibro, reproducir_audio

if os.path.exists("cuento.mp3"):
    os.remove("cuento.mp3")

await crear_audiolibro(cuento)

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
reproducir_audio("cuento.mp3")